# REM_Turku dream-affect decoding: baselines, nulls, and what they say

Handoff for **Paul Barbaste**. Runs top to bottom on Colab (GPU optional; the tangent
arms are fine on CPU). No cluster paths, no credentials.

Read this before any result, because each item below cost us a day:

1. **Permute labels WITHIN subject.** A global shuffle redraws each subject's base rate
   toward the grand mean and biases the null.
2. **Seed 0 means NO shuffle.** Null loops here assert `seed >= 1` so the unshuffled
   observed can never sit inside its own null.
3. **A permutation p at its floor is not a measurement.** Ours read 0.0175 at 56 draws
   and 0.0550 at 199. Report "p <= floor, not resolved" until draws resolve it.
4. **Metric level is not cosmetic.** The same 20 draws gave p <= 0.048 at epoch level and
   p = 0.1429 at awakening level for one observation, because epochs inside an awakening
   share its label. Every table row below is labelled with its level.


## 0. Configuration

`DATA_DIR` is the only path you should need to touch.


In [ ]:
import os, sys, subprocess, hashlib, json, zipfile, io, csv
from pathlib import Path

DATA_DIR = Path(os.environ.get("REMTURKU_DIR", "./remturku_data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
SEEDS_START_AT = 1          # seed 0 means no shuffle; never let it into a null loop


In [ ]:
%pip -q install pyriemann==0.12 braindecode mne scikit-learn scipy
# TSMNet arm only; comment out if you are not running it:
%pip -q install git+https://github.com/rkobler/TSMNet geoopt


## 1. Data

**We deliberately ship no preprocessed archive.** The prepared arrays are 2.8 GB, they
would dominate a Colab session, and shipping them would mean you trust our artefact
instead of verifying the pipeline. The cell below downloads the public deposit
(Sikka, Revonsuo, Noreika, Valli; figshare 10.6084/m9.figshare.23274596.v2, CC-BY 4.0,
363 MB) and rebuilds the epochs with the same script we ran (`prepare_remturku.py`,
sitting next to this notebook). First run ~15-20 min; cached afterwards.


In [ ]:
ZIP_URL = "https://figshare.com/ndownloader/articles/23274596/versions/2"
zip_path = DATA_DIR / "REM_Turku.zip"
npz_path = DATA_DIR / "remturku_epochs.npz"

if not npz_path.exists():
    if not zip_path.exists():
        print("downloading REM_Turku deposit (363 MB)...")
        subprocess.run(["curl", "-L", "-o", str(zip_path), ZIP_URL], check=True)
    here = Path("prepare_remturku.py")
    assert here.exists(), "run this notebook from the notebooks/ directory of the repo"
    subprocess.run([sys.executable, str(here), "--zip", str(zip_path),
                    "--out", str(npz_path)], check=True)
print("epochs ready:", npz_path, npz_path.exists())


## 2. The harness

One loader, one evaluator, one null. `evaluate(fit_predict, target)` takes any callable
`fit_predict(X_train, y_train, X_test) -> predictions` and scores it on the SAME
leave-one-subject-out folds every arm in the frozen table used. Both metric levels are
returned for every fold; nothing is silently mixed.


In [ ]:
import numpy as np
from scipy.stats import wilcoxon

HV = {"anger": ["SR_NA1", "SR_NA7"],
      "apprehension": ["SR_NA9", "SR_NA10"],
      "confusion": ["SR_PA2"]}

def _f(v):
    try:
        return float(v)
    except Exception:
        return None

def load(target):
    """Per-awakening raw epochs, binary label, subject id."""
    z = zipfile.ZipFile(DATA_DIR / "REM_Turku.zip")
    rat = {r["Filename"]: r for r in csv.DictReader(
        io.StringIO(z.read("REM_Turku/Data/Ratings.csv").decode("utf-8-sig")))}
    rec = {r["Filename"]: r for r in csv.DictReader(
        io.StringIO(z.read("REM_Turku/Records.csv").decode("utf-8-sig")))}
    npz = np.load(DATA_DIR / "remturku_epochs.npz")
    X, y, s = [], [], []
    for fn, r in rat.items():
        k = f"{fn}|raw"
        if k not in npz or fn not in rec:
            continue
        if not any((_f(r[c]) or 0) > 0 for c in r if c.startswith("SR_")):
            continue
        X.append(np.asarray(npz[k], dtype=np.float32))
        y.append(int(any((_f(r[c]) or 0) > 0 for c in HV[target])))
        s.append(rec[fn]["Subject ID"])
    return X, np.array(y), np.array(s)

def bal(p, t):
    p, t = np.asarray(p), np.asarray(t)
    tp = ((p == 1) & (t == 1)).sum(); fn = ((p == 0) & (t == 1)).sum()
    tn = ((p == 0) & (t == 0)).sum(); fp = ((p == 1) & (t == 0)).sum()
    se = tp / (tp + fn) if tp + fn else 0.0
    sp = tn / (tn + fp) if tn + fp else 0.0
    return float((se + sp) / 2)

def evaluate(fit_predict, target, shuffle_seed=0, verbose=True):
    """LOSO. Returns {subject: {'epoch': bal, 'awk': bal}} over scorable subjects.
    shuffle_seed >= 1 permutes labels WITHIN subject (for nulls); 0 = observed."""
    Xf, y_awk, s_awk = load(target)
    y_awk = y_awk.copy()
    if shuffle_seed:
        assert shuffle_seed >= 1, "seed 0 means NO shuffle; null loops start at 1"
        rng = np.random.default_rng(shuffle_seed)
        for u in np.unique(s_awk):
            m = s_awk == u
            y_awk[m] = rng.permutation(y_awk[m])
    # epoch-level arrays with awakening index for aggregation
    Xe, ye, se_, ae = [], [], [], []
    for i, (x, yy, ss) in enumerate(zip(Xf, y_awk, s_awk)):
        Xe.append(x); ye += [int(yy)] * len(x); se_ += [ss] * len(x); ae += [i] * len(x)
    Xe = np.concatenate(Xe); ye = np.array(ye)
    se_ = np.array(se_); ae = np.array(ae)
    out = {}
    for held in np.unique(se_):
        tr, te = se_ != held, se_ == held
        if len(np.unique(ye[te])) < 2:
            continue            # unscorable: single test class
        pe = np.asarray(fit_predict(Xe[tr], ye[tr], Xe[te]))
        pa, ta = [], []
        for aw in np.unique(ae[te]):
            m = ae[te] == aw
            pa.append(int(pe[m].mean() > 0.5)); ta.append(int(ye[te][m][0]))
        out[str(held)] = {"epoch": bal(pe, ye[te]), "awk": bal(pa, ta)}
        if verbose:
            print(f"held={held}  epoch={out[str(held)]['epoch']:.3f}  "
                  f"awk={out[str(held)]['awk']:.3f}")
    return out

def paired(res_a, res_b, level="awk"):
    """Wilcoxon over the shared held-out subjects. n = subjects, never epochs."""
    common = sorted(set(res_a) & set(res_b), key=int)
    a = np.array([res_a[s][level] for s in common])
    b = np.array([res_b[s][level] for s in common])
    return {"n": len(common), "mean_a": a.mean(), "mean_b": b.mean(),
            "delta_pp": 100 * (a - b).mean(), "wins": int((a > b).sum()),
            "p": float(wilcoxon(a, b).pvalue) if len(common) >= 6 else float("nan")}


## 3. The arms

Each arm is a factory returning `fit_predict`. Add yours at the bottom; everything else
stays untouched.


In [ ]:
def make_tangent_lda(recentre=False):
    from pyriemann.estimation import Covariances
    from pyriemann.tangentspace import TangentSpace
    from pyriemann.utils.mean import mean_riemann
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

    def _recentre(C):
        M = mean_riemann(C)
        w, V = np.linalg.eigh(M)
        Mi = V @ np.diag(1.0 / np.sqrt(np.maximum(w, 1e-12))) @ V.T
        return Mi @ C @ Mi

    def fit_predict(Xtr, ytr, Xte):
        cov = Covariances(estimator="oas")
        Ctr, Cte = cov.transform(Xtr), cov.transform(Xte)
        if recentre:
            Ctr, Cte = _recentre(Ctr), _recentre(Cte)   # label-free, legal under LOSO
        ts = TangentSpace().fit(Ctr)
        clf = LDA(solver="lsqr", shrinkage="auto").fit(ts.transform(Ctr), ytr)
        return clf.predict(ts.transform(Cte))
    return fit_predict


In [ ]:
def make_shallow_braindecode(epochs=30, lr=1e-3, batch=64, seed=0):
    """ShallowFBCSPNet at library defaults; the frozen table's DL reference."""
    import torch
    from braindecode.models import ShallowFBCSPNet

    def fit_predict(Xtr, ytr, Xte):
        torch.manual_seed(seed)
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        net = ShallowFBCSPNet(n_chans=Xtr.shape[1], n_outputs=2,
                              n_times=Xtr.shape[2]).to(dev)
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        lossf = torch.nn.CrossEntropyLoss()
        Xt = torch.from_numpy(Xtr).float(); yt = torch.from_numpy(ytr).long()
        idx = np.arange(len(Xt))
        net.train()
        for _ in range(epochs):
            np.random.shuffle(idx)
            for b in range(0, len(idx), batch):
                j = idx[b:b + batch]
                out = net(Xt[j].to(dev))
                loss = lossf(out, yt[j].to(dev))
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            preds = []
            for b in range(0, len(Xte), 256):
                out = net(torch.from_numpy(Xte[b:b + 256]).float().to(dev))
                preds.append(out.argmax(1).cpu().numpy())
        return np.concatenate(preds)
    return fit_predict


In [ ]:
def make_tsmnet(epochs=40, lr=1e-3, batch=256, seed=0):
    """TSMNet (Kobler et al., NeurIPS 2022), authoritative repo, defaults.

    DEVICE RULE, and the wrong fix named: TSMNet runs its SPD/tangent stage on CPU BY
    DESIGN (double precision), so bring the TARGET to the logits' device:
        loss = lossf(logits, y.to(logits.device))
    Do NOT walk the modules forcing every tensor to cuda: that makes the device
    crash vanish while silently moving the SPD stage off its intended device, and you
    get numbers from an architecture you did not mean to run."""
    import torch
    from spdnets.models import TSMNet
    from geoopt.optim import RiemannianAdam

    def fit_predict(Xtr, ytr, Xte):
        torch.manual_seed(seed)
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        X = np.concatenate([Xtr, Xte])
        X = (X - X.mean(-1, keepdims=True)) / (X.std(-1, keepdims=True) + 1e-7)
        Xtr_n, Xte_n = X[:len(Xtr)], X[len(Xtr):]
        net = TSMNet(temporal_filters=4, spatial_filters=40, subspacedims=20,
                     nclasses=2, nchannels=Xtr.shape[1], nsamples=Xtr.shape[2],
                     domains=torch.arange(2)).to(dev)
        opt = RiemannianAdam(net.parameters(), lr=lr)
        lossf = torch.nn.CrossEntropyLoss()
        Xt = torch.from_numpy(Xtr_n).float(); yt = torch.from_numpy(ytr).long()
        d0 = torch.zeros(len(Xt)).long()
        idx = np.arange(len(Xt))
        net.train()
        for _ in range(epochs):
            np.random.shuffle(idx)
            for b in range(0, len(idx), batch):
                j = idx[b:b + batch]
                out = net(Xt[j].to(dev), d0[j].to(dev))
                logits = out[0] if isinstance(out, tuple) else out
                loss = lossf(logits, yt[j].to(logits.device))
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            out = net(torch.from_numpy(Xte_n).float().to(dev),
                      torch.ones(len(Xte_n)).long().to(dev))
            logits = out[0] if isinstance(out, tuple) else out
            return logits.argmax(1).cpu().numpy()
    return fit_predict


## 4. Frozen results (2026-08-31), the table your method is scored against

Cross-subject LOSO, balanced accuracy, chance 0.50, n = scorable held-out subjects.
**Level** says which statistic the number is: `epoch` or `awakening` (the labelled unit;
the two can disagree, see trap 4).

| arm | level | apprehension | anger | confusion |
|---|---|---|---|---|
| TSMNet defaults (SPD) | awakening | **0.585 (14)**, null p=0.14 | 0.428 (16) | 0.513 (13) |
| TSMNet defaults (SPD) | epoch | 0.583 (14) | 0.435 (16) | 0.475 (13) |
| EEGNet defaults | epoch | 0.562 (14) | - | - |
| ShallowConv (bd) defaults | epoch | 0.546 (14) | 0.454 (16) | 0.507 (13) |
| ShallowConv nested-tuned | epoch | 0.557 (14) | - | 0.491 (13) |
| EEGNet nested-tuned | epoch | 0.530 (14) | 0.482 (16) | 0.542 (13) |
| tangent + LDA global | epoch | 0.552 (14) | 0.381 (16) | 0.482 (13) |
| tangent + LDA recentred | epoch | 0.512 (14) | 0.352 (16) | 0.437 (13) |

Significant results (each against its own null):

- **Tangent beats matched DL, paired: +2.9 pp, 11/14 subjects, Wilcoxon p = 0.012.**
- **TSMNet beats both tangent arms on anger, paired: p = 0.029 / 0.044.**
- **Anger decodes INVERTED: AUC 0.334 vs null 0.502, p = 0.010 (Bonferroni-passing)**,
  mechanism: between-subject r = +0.18, within-subject r = -0.24 (Simpson's paradox).
- Positive controls through the identical pipeline: subject identity 0.895 (17-way,
  null max 0.128), recording night 0.812 (null max 0.571). The nulls above are not a
  broken pipeline.

No arm clears chance on absolute dream-affect accuracy. **A method that does, under this
harness on these folds, is the headline of the paper.**


## 5. Run an arm

A full LOSO pass of the tangent arm takes a few minutes on CPU. The DL arms want the
Colab GPU. For a permutation null, call `evaluate` with `shuffle_seed = 1..N` and
compare the observed to the draw distribution; report a floor-limited p as a floor.


In [ ]:
res_tangent = evaluate(make_tangent_lda(), target="apprehension")
mean_awk = np.mean([v["awk"] for v in res_tangent.values()])
print(f"tangent+LDA apprehension: awakening-level mean {mean_awk:.4f} "
      f"over {len(res_tangent)} subjects")


## 6. Your entry point

Write a factory, run both cells, and compare paired. That is the whole integration.

```python
def make_your_method(**kw):
    def fit_predict(Xtr, ytr, Xte):
        ...
        return predictions
    return fit_predict

res_yours = evaluate(make_your_method(), target="apprehension")
print(paired(res_yours, res_tangent, level="awk"))
```

Then the permutation null (`shuffle_seed = 1..20` to start), and only then a claim.
`PREREG.md` in the repo root holds every pre-registration and the deviation log;
`COMMIT_MAP_2026-08-31.txt` maps pre-rewrite hashes cited there to current ones.
